# Análise de Reviews do Mercado Livre com GLiNER

Este notebook realiza uma análise automatizada de reviews de produtos do Mercado Livre utilizando o modelo GLiNER (Generalist and Lightweight model for Named Entity Recognition) para extração de entidades relevantes. O objetivo é identificar elementos como produtos mencionados, benefícios prometidos, características dos itens, elogios e críticas nos comentários dos usuários.

## Etapas do Notebook:

1. **Carregamento dos Dados**: Importação do dataset de reviews a partir de um arquivo CSV localizado em `../data/raw/reviews_mercadolivre.csv`.

2. **Inicialização do Modelo**: Carregamento do modelo GLiNER pré-treinado "numind/entity-recognition-gliner-large-v4", otimizado para tarefas de NER em português brasileiro.

3. **Definição de Labels**: Especificação das categorias de entidades a serem extraídas, incluindo "produto", "beneficio_prometido", "caracteristica_item", "elogio" e "critica".

4. **Processamento das Reviews**: Aplicação da função `processar_review` para extrair entidades de cada review, utilizando um threshold de 0.4 para balancear precisão e sensibilidade.

5. **Validação**: Teste em uma amostra de 20 reviews, com visualização dos resultados para verificar a eficácia da extração.

Este workflow permite insights automáticos sobre o feedback dos consumidores, facilitando a análise de sentimentos e características dos produtos.

In [4]:
import pandas as pd
from gliner import GLiNER
import os

# CONFIGURAÇÃO: escolha se quer trabalhar com a amostra ou com o arquivo completo
use_sample = True  # ajuste para False se quiser carregar o dataset inteiro

full_path = '../data/raw/reviews_mercadolivre.csv'
sample_path = '../data/processed/reviews_10pct_duckdb.csv'

if use_sample:
    # gera amostra via DuckDB se necessário
    if not os.path.exists(sample_path):
        try:
            import duckdb
            print('Gerando amostra de 10% via DuckDB...')
            total = duckdb.sql(f"SELECT COUNT(*) FROM read_csv_auto('{full_path}')").fetchone()[0]
            k = int(total * 0.1)
            duckdb.sql(f"""
            COPY (
              SELECT * FROM read_csv_auto('{full_path}')
              ORDER BY RANDOM()
              LIMIT {k}
            ) TO '{sample_path}' (HEADER);
            """)
            print(f'Amostra escrita em {sample_path} ({k} linhas)')
        except Exception as e:
            print('Falha ao gerar amostra:', e)
    df = pd.read_csv(sample_path)
else:
    df = pd.read_csv(full_path)

print(f"Dataset carregado ({'amostra' if use_sample else 'completo'}) com {len(df)} linhas")

# Inicializar o modelo GLiNER2
# O "large-v4" é o estado da arte para NER generalista e funciona muito bem em PT-BR
print("Carregando o modelo GLiNER2... (Isso pode levar um minuto)")
model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")

Dataset carregado (amostra) com 20695 linhas
Carregando o modelo GLiNER2... (Isso pode levar um minuto)


c:\Users\usuario\.conda\envs\geo-env\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

c:\Users\usuario\.conda\envs\geo-env\Lib\site-packages\transformers\convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [5]:
# Definimos o que queremos "caçar" nos comentários
labels = [
    "produto",             # Ex: Shampoo, Kit capilar
    "beneficio_prometido",  # Ex: hidratação, reconstrução
    "caracteristica_item",  # Ex: cheiro, textura, rendimento
    "elogio",               # Ex: maravilhoso, perfeito
    "critica"               # Ex: demora, caro, não funcionou
]

def processar_review(texto):
    if pd.isna(texto) or texto == "":
        return []
    # threshold 0.4 é um bom equilíbrio entre precisão e sensibilidade
    entities = model.predict_entities(texto, labels, threshold=0.4)
    return entities

## Validação e Execução

Aqui realizamos uma validação inicial com uma amostra pequena, seguida do processamento do dataset completo e da amostra de 10% gerada com DuckDB.

In [6]:
# Pegamos uma amostra de 20 reviews para validar
df_teste = df.sample(50).copy()

print("Iniciando extração com IA...")
df_teste['analise_ia'] = df_teste['content'].apply(processar_review)

# Visualizar o resultado de forma legível
for idx, row in df_teste.iterrows():
    print(f"REVIEW: {row['content']}")
    if row['analise_ia']:
        for ent in row['analise_ia']:
            print(f"  -> [{ent['label'].upper()}]: {ent['text']}")
    else:
        print("  -> Nenhuma entidade encontrada.")
    print("-" * 20)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Iniciando extração com IA...
REVIEW: Adoro produto dessa linha , o infantil também é top deixa o cabelo macio e cheiroso!.
  -> [PRODUTO]: o infantil
--------------------
REVIEW: Achei excelente nos amamos, bem melhor do que famoso da marca seda.
Pois seda não tira 100% oleosos de cabelos e o produto aqui mercadolivre bem superior lavar bem tira totalmente sem oleoso.
  -> [PRODUTO]: seda
  -> [PRODUTO]: seda
  -> [PRODUTO]: mercadolivre
--------------------
REVIEW: Fiquei muito chateada. Comprei o kit azul e veio o anti-queda. Devim ter avisado antes.
  -> [PRODUTO]: kit azul
  -> [PRODUTO]: anti-queda
--------------------
REVIEW: Produto ótimo desmaio meu cabelo indico não fazer botox no dia porque ele e ótimo mesmo devolvem brilho macieis deixou super hidratado mais peso depois que lavei fico ótimo leve e hidratado 😍.
  -> [PRODUTO]: Produto
--------------------
REVIEW: Selagem , excelente! amo.
Alisa perfeitamente,afro como loiro!.
  -> [CARACTERISTICA_ITEM]: amo
------------------

In [ ]:
# Processar todo o dataset em blocos com log de progresso e estimativa de tempo
print(f"Processando {len(df)} reviews completos... Isso pode levar alguns minutos.")

import time

# definimos um batch size para impressão de progresso
def batch_process(texts, batch_size=200):
    results = []
    total = len(texts)
    start_time = time.time()
    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        chunk = texts[start:end]
        # chamada única para o modelo sobre o lote
        batch_entities = [processar_review(txt) for txt in chunk]
        results.extend(batch_entities)
        elapsed_reviews = start + len(chunk)
        elapsed_time = time.time() - start_time
        avg_per_review = elapsed_time / elapsed_reviews
        remaining = total - elapsed_reviews
        eta = remaining * avg_per_review
        print(f"  > Processados {elapsed_reviews}/{total} reviews ({elapsed_reviews/total:.1%}) - ETA {eta/60:.1f} min")
    return results

# aplicar batch processing e armazenar
df['analise_ia'] = batch_process(df['content'].tolist(), batch_size=200)

# Salvar os dados processados
os.makedirs('../data/processed', exist_ok=True)
output_path = '../data/processed/reviews_com_entidades.csv'
df.to_csv(output_path, index=False, encoding='utf-8')
print(f"Dados processados salvos em: {output_path}")
print(f"Total de reviews processados: {len(df)}")
print(f"Colunas finais: {list(df.columns)}")

# Estatísticas rápidas
total_entidades = df['analise_ia'].apply(len).sum()
print(f"Total de entidades extraídas: {total_entidades}")
print(f"Média de entidades por review: {total_entidades / len(df):.2f}")

In [9]:
# Caso já tenhamos gerado o arquivo de amostra com DuckDB, podemos carregá‑lo e processá‑lo
# esta rotina define sua própria versão de batch_process para não depender de células anteriores

import time

def processar_duckdb_sample(path='../data/processed/reviews_10pct_duckdb.csv'):
    if not os.path.exists(path):
        print(f"Arquivo de amostra não encontrado: {path}")
        return None
    print(f"Carregando amostra do DuckDB em {path}...")
    df_sample = pd.read_csv(path)
    print(f"Processando {len(df_sample)} linhas da amostra...")
    
    # definição local de batch_process para garantir disponibilidade
    def batch_process(texts, batch_size=200):
        results = []
        total = len(texts)
        start_time = time.time()
        for start in range(0, total, batch_size):
            end = min(start + batch_size, total)
            chunk = texts[start:end]
            batch_entities = [processar_review(txt) for txt in chunk]
            results.extend(batch_entities)
            elapsed_reviews = start + len(chunk)
            elapsed_time = time.time() - start_time
            avg_per_review = elapsed_time / elapsed_reviews
            remaining = total - elapsed_reviews
            eta = remaining * avg_per_review
            print(f"  > Processados {elapsed_reviews}/{total} reviews ({elapsed_reviews/total:.1%}) - ETA {eta/60:.1f} min")
        return results
    
    df_sample['analise_ia'] = batch_process(df_sample['content'].tolist(), batch_size=200)
    out = path.replace('.csv', '_com_entidades.csv')
    df_sample.to_csv(out, index=False, encoding='utf-8')
    print(f"Amostra processada salva em: {out}")
    return df_sample

# exemplo de uso:
df_duck = processar_duckdb_sample()

Carregando amostra do DuckDB em ../data/processed/reviews_10pct_duckdb.csv...
Processando 20695 linhas da amostra...
  > Processados 200/20695 reviews (1.0%) - ETA 87.9 min
  > Processados 400/20695 reviews (1.9%) - ETA 84.9 min
  > Processados 600/20695 reviews (2.9%) - ETA 82.3 min
  > Processados 800/20695 reviews (3.9%) - ETA 80.0 min
  > Processados 1000/20695 reviews (4.8%) - ETA 77.9 min
  > Processados 1200/20695 reviews (5.8%) - ETA 74.9 min
  > Processados 1400/20695 reviews (6.8%) - ETA 72.8 min
  > Processados 1600/20695 reviews (7.7%) - ETA 72.2 min
  > Processados 1800/20695 reviews (8.7%) - ETA 70.9 min
  > Processados 2000/20695 reviews (9.7%) - ETA 70.2 min
  > Processados 2200/20695 reviews (10.6%) - ETA 69.8 min
  > Processados 2400/20695 reviews (11.6%) - ETA 69.1 min
  > Processados 2600/20695 reviews (12.6%) - ETA 68.5 min
  > Processados 2800/20695 reviews (13.5%) - ETA 68.2 min
  > Processados 3000/20695 reviews (14.5%) - ETA 67.4 min
  > Processados 3200/20695 